In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd


# Read CSV file
df = pd.read_csv("Q1_data.csv")
# Display first few rows
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df["delivery_time"], bins=30, kde=True)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()


In [ ]:
df = df.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
df.isnull().sum()
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Fill numerical missing values with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
# Check duplicates
df.duplicated().sum()
# Remove duplicates if any
df = df.drop_duplicates()


In [ ]:
# One Hot Encoding for categorical variables
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["delivery_time"])
y = df["delivery_time"]


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X = df.drop(columns=["delivery_time"])
y = df["delivery_time"]


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
from sklearn.model_selection import KFold
# KFold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train model
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluate using MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
print(f"Average MAE across all folds: {np.mean(mae_scores):.2f}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot
plt.figure(figsize=(10,6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)),
           feature_names[indices],
           rotation=90)
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Feature Importance - Random Forest")
plt.tight_layout()
plt.show()


In [ ]:
# Predict delivery time
y_pred = final_model.predict(X)

# Plot histogram
plt.figure(figsize=(8,5))
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Histogram of Predicted Delivery Time")
plt.show()


In [ ]:
#maybe vallue is changei will make it from my mind
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from catboost import CatBoostRegressor


# Load data

df = pd.read_csv("Q1_data.csv")


# Pick target column (robust default)

# If you have a specific target name, set it here, e.g. target_col = "SalePrice"
candidate_targets = ["target", "Target", "y", "Y", "label", "Label"]
target_col = next((c for c in candidate_targets if c in df.columns), df.columns[-1])

X = df.drop(columns=[target_col])
y = df[target_col]

# -----------------------------
# Identify column types
# -----------------------------
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Model 1: RandomForest

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

rf_model = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])


# Model 2: CatBoost handle categorical
# CatBoost
cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols]

cb_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="MAE",
    random_seed=42,
    verbose=0
)


# kfold ensemble  : average preds + mae

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Fit RF
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)

    # Fit CatBoost
    cb_model.fit(X_train, y_train, cat_features=cat_feature_indices)
    cb_pred = cb_model.predict(X_val)

    # Average predictions
    avg_pred = (rf_pred + cb_pred) / 2.0

    # MAE on averaged predictions
    fold_mae = mean_absolute_error(y_val, avg_pred)
    mae_scores.append(fold_mae)

    print(f"Fold {fold} MAE (ensemble): {fold_mae:.6f}")

print("\nEnsemble MAE (mean):", np.mean(mae_scores))
print("Ensemble MAE (std): ", np.std(mae_scores))
